In [1]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, T5ForConditionalGeneration, T5Tokenizer
import torch
import json

In [2]:
with open("Model_dataset/cv.json", "r") as file:
    CV_DATA= json.load(file)

In [3]:
q_type_model= 'model/fine_tuned_question_classifier_model_lite-AdamW'
qa_type_model= 't5-large'

In [4]:
QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_model)
QUESTION_CLASSIFIER_MODEL.eval()
QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_model)
ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_model)
QUESTION_ANSWER_MODEL.eval()
QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_model, legacy= False)


In [5]:
def get_question_type_prediction(text):
    inputs = QUESTION_CLASSIFIER_TOKENIZER(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
    logits = outputs.logits
    predicted_classes = torch.argmax(logits, dim=1)
    id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
    return id2label[predicted_classes.item()]
# get_question_type_prediction("How much do you want to earn?")

In [6]:
def get_model_out_raw(question):
    predicted_question_type= get_question_type_prediction(question)
    context= CV_DATA[predicted_question_type]
    input_text = f"question: {question} context: {context}"
    inputs = QUESTION_ANSWER_TOKENIZER(input_text, return_tensors="pt")
    outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)

    return predicted_question_type, QUESTION_ANSWER_TOKENIZER.decode(outputs[0], skip_special_tokens=True)


# get_model_out_raw("Total work experince in python?")